# 🎙️ 長時間会議 自動文字起こし・議事録作成システム

## 使い方
1. `MeetingTranscript/01_input` フォルダに音声ファイルを入れる
2. スプレッドシートの「本日の参加者」を更新する
3. **「すべてのセルを実行」を押して待つだけ** ☕

---

## Step 1: 環境セットアップ
必要なライブラリをインストールします（初回のみ数分かかります）

In [ ]:
# ライブラリのインストール
# ※ 初回実行時は数分かかります

# transformers のバージョンを whisperx に合わせて固定
!pip install -q "transformers==4.44.2"
!pip install -q whisperx
!pip install -q gspread google-auth
!pip install -q google-generativeai
!pip install -q notion-client

print('✅ ライブラリのインストール完了')

In [ ]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive のマウント完了')

In [ ]:
# APIキーの読み込み（Colab のシークレット機能を使用）
# ※ 左サイドバー 🔑 アイコン → シークレットに以下を登録してください
#   - HF_TOKEN       : Hugging Face のトークン
#   - GEMINI_API_KEY : Gemini API キー
#   - NOTION_TOKEN   : Notion インテグレーションのトークン

from google.colab import userdata

HF_TOKEN       = userdata.get('HF_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NOTION_TOKEN   = userdata.get('NOTION_TOKEN')

# 登録漏れがないか確認
missing = [name for name, val in [
    ('HF_TOKEN', HF_TOKEN),
    ('GEMINI_API_KEY', GEMINI_API_KEY),
    ('NOTION_TOKEN', NOTION_TOKEN)
] if not val]

if missing:
    raise ValueError(f'❌ 以下のシークレットが未登録です: {missing}')

print('✅ APIキーの読み込み完了')

In [ ]:
# ========================================
# ⚙️ 設定（ここだけ自分の環境に合わせて変更）
# ========================================

# Google Drive 上のフォルダパス
BASE_DIR        = '/content/drive/MyDrive/MeetingTranscript'
INPUT_AUDIO_DIR = f'{BASE_DIR}/01_input'
DONE_AUDIO_DIR  = f'{BASE_DIR}/02_processed'
TEXT_OUTPUT_DIR = f'{BASE_DIR}/03_output'

# スプレッドシートの設定
SPREADSHEET_NAME = 'MeetingTranscript'  # スプレッドシートの名前（04_management フォルダ内）
WORKSHEET_NAME   = '本日の参加者'        # シート名

# Notion の設定
NOTION_DATABASE_ID = 'ここにNotionのデータベースIDを入力'  # 例: 'abc123def456...'

# Gemini の設定
GEMINI_MODEL         = 'gemini-3.1-flash-lite'  # 無料枠が最も多い（RPD: 500）
SPEAKER_SAMPLE_CHARS = 3000   # 話者特定に使う冒頭文字数
CHUNK_SIZE           = 10000  # 整形処理の分割単位（文字数）

print('✅ 設定の読み込み完了')
print(f'   入力フォルダ : {INPUT_AUDIO_DIR}')
print(f'   出力フォルダ : {TEXT_OUTPUT_DIR}')

---
## Step 2: 音声ファイルの文字起こし（WhisperX）
GPUを使って音声を文字起こし＋話者分離します

In [ ]:
import os
import glob

# 入力フォルダから音声ファイルを1本取得
audio_files = glob.glob(f'{INPUT_AUDIO_DIR}/*.mp3') + \
              glob.glob(f'{INPUT_AUDIO_DIR}/*.m4a') + \
              glob.glob(f'{INPUT_AUDIO_DIR}/*.wav')

if not audio_files:
    raise FileNotFoundError(f'❌ {INPUT_AUDIO_DIR} に音声ファイルがありません')

# 最初の1ファイルを処理対象にする
audio_path = audio_files[0]
print(f'✅ 処理対象ファイル: {os.path.basename(audio_path)}')

In [ ]:
import whisperx

# デバイスの設定（GPUが使える場合はcuda、使えない場合はcpu）
device = 'cuda'
compute_type = 'float16'

print('🎙️ WhisperX で文字起こし中... (しばらくかかります)')

# モデルの読み込み（largeモデルで高精度）
model = whisperx.load_model('large-v3', device, compute_type=compute_type)

# 文字起こし実行
audio = whisperx.load_audio(audio_path)
result = model.transcribe(audio, batch_size=16, language='ja')

print(f'✅ 文字起こし完了（{len(result["segments"])} セグメント）')

In [ ]:
# 話者分離（誰が話しているか識別）
print('👥 話者分離中...')

# 音素アライメント
model_a, metadata = whisperx.load_align_model(language_code='ja', device=device)
result = whisperx.align(result['segments'], model_a, metadata, audio, device)

# 話者ラベルの付与
diarize_model = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

print('✅ 話者分離完了')

In [ ]:
# 文字起こし結果を「SPEAKER_XX: テキスト」形式に整形
transcript_lines = []
for segment in result['segments']:
    speaker = segment.get('speaker', 'SPEAKER_UNKNOWN')
    text    = segment['text'].strip()
    if text:  # 空行はスキップ
        transcript_lines.append(f'[{speaker}] {text}')

# テキスト全文を結合
full_transcript = '\n'.join(transcript_lines)

# 一時保存（途中でColabが落ちても安心）
os.makedirs(TEXT_OUTPUT_DIR, exist_ok=True)
raw_text_path = f'{TEXT_OUTPUT_DIR}/raw_transcript.txt'
with open(raw_text_path, 'w', encoding='utf-8') as f:
    f.write(full_transcript)

print(f'✅ 生テキストを保存: {raw_text_path}')
print(f'   総文字数: {len(full_transcript)} 文字')
print(f'   冒頭サンプル:')
print(full_transcript[:300])

---
## Step 3: スプレッドシートから参加者情報を取得

In [ ]:
import gspread
from google.auth import default

# Google 認証（Colab 環境では自動で認証される）
creds, _ = default()
gc = gspread.authorize(creds)

# スプレッドシートを開く
spreadsheet = gc.open(SPREADSHEET_NAME)
worksheet   = spreadsheet.worksheet(WORKSHEET_NAME)

# 参加者データを取得（ヘッダー行を除く）
records = worksheet.get_all_records()

if not records:
    raise ValueError('❌ スプレッドシートに参加者情報がありません。更新してください。')

# 参加者情報を確認
print('✅ 参加者情報を取得しました:')
for row in records:
    print(f"   {row['名前']}（{row['役割']}）: {row['プロフィール']}")

---
## Step 4: 話者特定（Gemini）→ 名前置換（Python）→ テキスト整形

In [ ]:
import google.generativeai as genai
import json
import re

# Gemini の初期化
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel(GEMINI_MODEL)

# --- 話者特定 ---
# 参加者プロフィールをテキスト化
profile_text = '\n'.join([
    f"- {row['名前']}（{row['役割']}）: {row['プロフィール']}"
    for row in records
])

# 冒頭3000文字を抽出
sample_text = full_transcript[:SPEAKER_SAMPLE_CHARS]

# Gemini に話者特定を依頼
identify_prompt = f"""以下の会議の文字起こし（冒頭部分）と参加者情報を元に、
SPEAKER_XX が誰に対応するか特定してください。

【参加者情報】
{profile_text}

【文字起こし（冒頭）】
{sample_text}

必ず以下のJSON形式のみで回答してください。余分なテキストは不要です。
{{"SPEAKER_00": "名前", "SPEAKER_01": "名前", ...}}
"""

print('🤖 Gemini で話者を特定中...')
response = gemini.generate_content(identify_prompt)

# JSON部分だけ抽出してパース
json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
if not json_match:
    raise ValueError(f'❌ Gemini のレスポンスからJSONを取得できませんでした:\n{response.text}')

speaker_map = json.loads(json_match.group())
print('✅ 話者の対応表:')
for speaker_id, name in speaker_map.items():
    print(f'   {speaker_id} → {name}')

In [ ]:
# --- Python で機械的に名前を置換 ---
replaced_transcript = full_transcript
for speaker_id, name in speaker_map.items():
    replaced_transcript = replaced_transcript.replace(f'[{speaker_id}]', f'{name}:')

# マッピングされなかった SPEAKER_XX が残っていないか確認
remaining = re.findall(r'\[SPEAKER_\d+\]', replaced_transcript)
if remaining:
    print(f'⚠️ 未置換の話者が残っています: {set(remaining)}')
    print('   スプレッドシートの参加者情報を確認してください')
else:
    print('✅ 全話者の名前置換が完了しました')

In [ ]:
# --- Gemini で専門用語補正＋対話形式に整形 ---
# テキストを1万文字ずつ分割して処理
def split_text(text, chunk_size):
    """テキストを改行単位で chunk_size 文字以内に分割する"""
    chunks = []
    current_chunk = []
    current_len = 0

    for line in text.split('\n'):
        line_len = len(line) + 1  # 改行分を加算
        if current_len + line_len > chunk_size and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = []
            current_len = 0
        current_chunk.append(line)
        current_len += line_len

    if current_chunk:
        chunks.append('\n'.join(current_chunk))

    return chunks

chunks = split_text(replaced_transcript, CHUNK_SIZE)
print(f'✅ テキストを {len(chunks)} チャンクに分割しました')

# 各チャンクを Gemini で整形
formatted_chunks = []
for i, chunk in enumerate(chunks):
    print(f'🤖 チャンク {i+1}/{len(chunks)} を整形中...')

    format_prompt = f"""以下は会議の文字起こしです。
専門用語の明らかな誤認識を補正し、読みやすい対話形式に整えてください。

ルール:
- 話者名と発言内容はそのまま保持する（例: 田中: 〜〜〜）
- 意味が変わるような要約・削除はしない（全文を保持）
- 明らかな誤字・誤変換のみ修正する

【文字起こし】
{chunk}
"""

    response = gemini.generate_content(format_prompt)
    formatted_chunks.append(response.text)

# 全チャンクを結合
final_text = '\n\n'.join(formatted_chunks)
print(f'✅ テキスト整形完了（総文字数: {len(final_text)} 文字）')

---
## Step 5: Notion に議事録ページを作成

In [ ]:
from notion_client import Client
from datetime import date

notion = Client(auth=NOTION_TOKEN)

# 本日の日付とページタイトルを生成
today = date.today().strftime('%Y-%m-%d')
page_title = f'議事録_{today}'

# 参加者リストを文字列化
participants = '、'.join([
    f"{row['名前']}（{row['役割']}）" for row in records
])

# Notion のブロック形式に変換（100ブロックずつ分割して送信）
def text_to_blocks(text):
    """テキストを Notion の段落ブロックのリストに変換する"""
    blocks = []
    for line in text.split('\n'):
        blocks.append({
            'object': 'block',
            'type': 'paragraph',
            'paragraph': {
                'rich_text': [{'type': 'text', 'text': {'content': line}}]
            }
        })
    return blocks

print(f'📝 Notion にページを作成中: {page_title}')

# ページを作成（タイトルのみ）
new_page = notion.pages.create(
    parent={'database_id': NOTION_DATABASE_ID},
    properties={
        'タイトル': {
            'title': [{'text': {'content': page_title}}]
        }
    }
)

page_id = new_page['id']

# 本文の冒頭に日付・参加者情報を追加
header_blocks = [
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {
            'rich_text': [{'type': 'text', 'text': {'content': f'日付: {today}'}}]
        }
    },
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {
            'rich_text': [{'type': 'text', 'text': {'content': f'参加者: {participants}'}}]
        }
    },
    {
        'object': 'block',
        'type': 'divider',
        'divider': {}
    }
]
notion.blocks.children.append(block_id=page_id, children=header_blocks)

# 本文を100ブロックずつに分けて追加（Notion API の制限）
all_blocks = text_to_blocks(final_text)
for i in range(0, len(all_blocks), 100):
    batch = all_blocks[i:i+100]
    notion.blocks.children.append(block_id=page_id, children=batch)
    print(f'   ブロック追加中... {min(i+100, len(all_blocks))}/{len(all_blocks)}')

page_url = new_page['url']
print(f'✅ Notion ページの作成完了！')
print(f'   🔗 {page_url}')

---
## Step 6: 後片付け

In [ ]:
import shutil
from datetime import datetime

# 処理済み音声ファイルを移動
os.makedirs(DONE_AUDIO_DIR, exist_ok=True)
done_path = os.path.join(DONE_AUDIO_DIR, os.path.basename(audio_path))
shutil.move(audio_path, done_path)
print(f'✅ 音声ファイルを移動: {done_path}')

# 最終テキストを保存
final_text_path = f'{TEXT_OUTPUT_DIR}/minutes_{today}.md'
with open(final_text_path, 'w', encoding='utf-8') as f:
    f.write(f'# {page_title}\n\n')
    f.write(f'参加者: {participants}\n\n')
    f.write('---\n\n')
    f.write(final_text)
print(f'✅ 議事録を保存: {final_text_path}')

# 処理ログをスプレッドシートに記録
try:
    log_sheet = spreadsheet.worksheet('処理ログ')
except gspread.WorksheetNotFound:
    log_sheet = spreadsheet.add_worksheet(title='処理ログ', rows=1000, cols=5)
    log_sheet.append_row(['日時', 'ファイル名', '参加者', 'Notionページ'])

log_sheet.append_row([
    datetime.now().strftime('%Y-%m-%d %H:%M'),
    os.path.basename(done_path),
    participants,
    page_url
])
print('✅ 処理ログを記録しました')

print()
print('=' * 50)
print('🎉 すべての処理が完了しました！')
print(f'   Notion ページ: {page_url}')
print('=' * 50)